# 11. Lecke - Modell Kontextus Protokoll (MCP)

A **Modell Kontextus Protokoll (MCP)** egy nyílt szabvány, amely lehetővé teszi az ügynökök számára, hogy futásidőben dinamikusan felfedezzenek és használjanak eszközöket, erőforrásokat és adatforrásokat. Ahelyett, hogy az eszközöket keménykódolnánk egy ügynökbe, az MCP lehetővé teszi az ügynököknek, hogy külső szerverekhez csatlakozzanak, amelyek igény szerint kínálnak képességeket.

Ebben a leckében megtanulod:
- Mi az MCP és miért fontos az ügynök rendszerek számára
- Hogyan működik az MCP kliens-szerver architektúrája
- Hogyan építsünk olyan ügynököket, amelyek MCP-stílusú eszközfelfedezést használnak


## Beállítás

**Előfeltételek:**
- Microsoft Foundry projekt egy telepített modellel
- Az azonosításhoz futtassa az `az login` parancsot


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Mi az a Model Context Protocol (MCP)?

Az MCP egy szabványos módot határoz meg arra, hogy az AI ügynökök felfedezzék és kapcsolatba lépjenek külső eszközökkel és adatforrásokkal:

- **MCP szerver**: Eszközöket, erőforrásokat és promptokat tesz elérhetővé egy szabványos protokollon keresztül
- **MCP kliens**: Az ügynök futtatási környezete, amely kapcsolódik a szerverekhez és felfedezi a rendelkezésre álló képességeket
- **Dinamikus felfedezés**: Az ügynököknek nem kell keménykódolt eszközök — futás közben fedezik fel, mi érhető el

Ez erőteljes megoldás bővíthető ügynökrendszerek építéséhez, ahol új képességek hozzáadhatók az ügynök kódjának módosítása nélkül.


## Hogyan működik az MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Az ügynök (MCP kliens) csatlakozik egy MCP szerverhez
2. A szerver válaszként listát küld az elérhető eszközökről és azok sémáiról
3. Az ügynök ezután a gondolkodása során bármelyik felfedezett eszközt meghívhatja
4. Az eredmények ugyanazon protokollon keresztül érkeznek vissza


## MCP eszközfelismerés szimulálása

Mivel egy valódi MCP szerverhez futó szerverfolyamat szükséges, a mintát `@tool` függvények segítségével mutatjuk be, amelyek azt szimulálják, amit egy MCP-hez kapcsolódó elhelyezési szolgáltatás nyújtana.

Éles környezetben ezeket az eszközöket dinamikusan fedezné fel az MCP szerver, ahelyett, hogy helyileg lennének definiálva.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Ügynök létrehozása MCP-stílusú eszközökkel


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP a gyártásban

Egy éles környezetben az MCP lehetővé teszi az erőteljes mintákat:

- **Dinamikus eszközfelismerés**: Az ügynökök kapcsolódnak az MCP szerverekhez és futásidőben fedezik fel az eszközöket
- **Lecsatolt architektúra**: Az eszközszolgáltatók függetlenül frissíthetik magukat az ügynöktől
- **Szervezetek közötti megosztás**: Csapatok képességeket tehetnek elérhetővé MCP szervereken keresztül, amelyeket bármely ügynök használhat
- **Microsoft Agent Framework támogatás**: Az MAF beépített MCP kliens támogatással rendelkezik a `mcp` integráción keresztül

Ahhoz, hogy valódi MCP szervert használjon az MAF-fal, a `hosted_mcp_tool()` vagy az MCP kliens integráció révén kell kapcsolódnia.

**További információ:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Összefoglaló

Ebben a leckében megtanultad:
- Az **MCP** egy nyílt szabvány az eszközök dinamikus felfedezésére ügynökök és eszközszolgáltatók között
- A **kliens-szerver architektúra** lehetővé teszi az ügynököknek, hogy futásidőben fedezzék fel a képességeket
- Az MCP lehetővé teszi a **kiterjeszthető, leválasztott ügynökrendszerek** kialakítását, ahol az eszközök kódmódosítás nélkül adhatók hozzá
- A Microsoft Agent Framework **beépített MCP támogatást** nyújt éles használatra


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
